# MEng Thesis Figures/Results Notebook

This notebook was used to produce the figures for my MEng thesis which will hopefully eventually be made publicly available. As the content of the thesis is effectively a superset of the CTDS preprint on arXiv, this notebook accordingly extends `notebooks/camera_ready_figures.ipynb` by additionally including library-specific visualizations of e.g., Langevin dynamics and various annealing paths, as well as NETS and CTDS using the microcanonical Langevin dynamics given by the energy sampling Hamiltonian [1].

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import math
sys.path.append('../')

import torch
from omegaconf import OmegaConf
from matplotlib import pyplot as plt
from hydra import compose, initialize

from src.models.nets import NETSModule
from src.models.ctds import CTDSModule
from src.simulation.dynamics import ForwardProcessSampleable
from src.utils.misc import get_device
from src.utils.plotting import plot_proposal, mt_plot_proposal, timeseries_plot_density, timeseries_plot_contours, mt_timeseries_plot_density, mt_timeseries_plot_contours

device = get_device()
device = torch.device('cuda:2')

print(f'Using device: {device}')

# Part 1: General Visualizations

### Figure 1: Comparison of Linear, Learned, and Interpolant Density Paths

In [ ]:
# Load PINN module w/ linear path
linear_path = '../pretrained_checkpoints/linear/baseline.ckpt'
linear_pinn = NETSModule.load_from_checkpoint(linear_path).move_to_device(device)

# Load PINN module w/ learned path
overdamped_jarzynski_path = '../pretrained_checkpoints/learned/nets_overdamped_jarzynski.ckpt'
learned_pinn = NETSModule.load_from_checkpoint(overdamped_jarzynski_path).move_to_device(device)

# Load PINN module w/ interpolant path
os.environ["RUN_CONFIG"] = "interpolant_esh" 
with initialize(version_base=None, config_path="../config"):
    interpolant_cfg = compose(config_name="nets_config")

interpolant_pinn = NETSModule(cfg=interpolant_cfg.pinn_config).move_to_device(device)

# Create ts
ts = torch.linspace(0, 1, 5).to(device)
scale = 50.0

# Create plots
fig, axes = plt.subplots(3, len(ts), figsize=(20, 10), dpi=500)

# Loop through axes and set ticks to empty
for i in range(3):
    for j in range(len(ts)):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

# Set left-most ticks to zero
for i in range(3):
    axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
    axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=8)
    pass

# Set bottom-most ticks to zero
for i in range(len(ts)):
    axes[2, i].set_xticks([-40, -20, 0, 20, 40])
    axes[2, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=8)
    axes[2, i].set_xlabel(f't={ts[i].item():.2f}', fontsize=15)

axes[0, 0].set_ylabel('Linear Path', fontsize=15)
axes[1, 0].set_ylabel('Learned Path', fontsize=15)
axes[2, 0].set_ylabel('Interpolant Path', fontsize=15)

axes = axes.reshape((3, len(ts)))

# Plot linear path and contours
timeseries_plot_density(linear_pinn.density_path.log_density, ts, scale=scale, axes=axes[0], alpha=0.5, vmin=-20, zorder=0, device=device)
timeseries_plot_contours(linear_pinn.density_path.log_density, ts, scale=scale, axes=axes[0], alpha=0.5, legend=False, levels=20, device=device)

# Plot learned path and contours
timeseries_plot_density(learned_pinn.density_path.log_density, ts, scale=scale, axes=axes[1], alpha=0.5, vmin=-20, zorder=0, device=device)
timeseries_plot_contours(learned_pinn.density_path.log_density, ts, scale=scale, axes=axes[1], alpha=0.5, legend=False, levels=20, device=device)

# Plot interpolant path and contours
timeseries_plot_density(interpolant_pinn.density_path.log_density, ts, scale=scale, axes=axes[2], alpha=0.5, vmin=-20, zorder=0, device=device)
timeseries_plot_contours(interpolant_pinn.density_path.log_density, ts, scale=scale, axes=axes[2], alpha=0.5, legend=False, levels=20, device=device)

plt.show()

### Figure 2: Langevin Dynamics

In [ ]:
from src.systems.density_path import ConstantDensityPath
from src.systems.distributions import Gaussian, GMM
from src.simulation.langevin import AnnealedOverdampedLangevin
from src.simulation.vector_field import ZeroVectorField
from src.utils.plotting import plot_process


# Simulation
scale=50.0
num_timesteps = 501
num_marginals = 3

# Source
x_source = Gaussian.isotropic(dim=2, std=20.0).to(device)

# Target 
density = GMM.FAB_GMM(cov_scale=0.25).to(device)

# Density Path
density_path = ConstantDensityPath(
    sampleable=x_source,
    density=density,
).to(device)

# Dynamics 
damping = 1.0
process = AnnealedOverdampedLangevin(
    control=ZeroVectorField(out_dim=2).to(device),
    damping=damping,
    density_path=density_path,
).to(device)

ts = torch.linspace(0, 1, num_timesteps).to(device)
axes, plotted_timesteps = plot_process(
    process=process,
    num_samples=1000,
    timesteps=ts,
    record_every=num_timesteps // num_marginals,
    scale=scale,
)

# Reset the axes
for step in range(num_marginals + 1):
    ax = axes[step]
    t = plotted_timesteps[step].item()
    ax.set_xticks([-40, -20, 0, 20, 40])
    ax.set_xticklabels([-40, -20, 0, 20, 40], fontsize=20)
    ax.set_xlabel(f't={t:.2f}', fontsize=20)
    ax.set_title('')

axes[0].set_yticks([-40, -20, 0, 20, 40])
axes[0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=20)

timeseries_plot_density(density_path.log_density, plotted_timesteps, scale=scale, axes=axes, alpha=0.5, vmin=-40, zorder=0, device=device)
timeseries_plot_contours(density_path.log_density, plotted_timesteps, scale=scale, axes=axes, alpha=0.5, legend=False, levels=20, device=device)

### Figure 4: Density Continuum

In [ ]:
# Load
os.environ["RUN_CONFIG"] = "learned_tempered" 
with initialize(version_base=None, config_path="../config"):
    cfg = compose(config_name="ctds_config")
    cfg.mt_pinn_config.mt_continuum = 'linear'

mt_pinn = CTDSModule(cfg=cfg.mt_pinn_config).move_to_device(device)

# Graph
mt_pinn.proposal.record_every = 100
t_bins = torch.linspace(0, 1, 5).to(device)
b_bins = torch.linspace(0.2, 1.0, 3).to(device)
scale = 50.0
fig, axes = plt.subplots(len(b_bins), len(t_bins), figsize=(20, 10), dpi=500)

# Title the axes and set ticks to empty
for i in range(len(b_bins)):
    for j in range(len(t_bins)):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

# Set border ticks on temp axis
for i in range(len(b_bins)):
    axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
    axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=8)
    axes[i, 0].set_ylabel(r'$\beta=$' + f'{b_bins[i].item():.2f}', fontsize=15)

# Set border ticks on time axis
for i in range(len(t_bins)):
    axes[len(b_bins)-1, i].set_xticks([-40, -20, 0, 20, 40])
    axes[len(b_bins)-1, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=8)
    axes[len(b_bins)-1, i].set_xlabel(f't={t_bins[i].item():.2f}', fontsize=15)

mt_timeseries_plot_density(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, zorder=0, vmin=-40, device=device)
mt_timeseries_plot_contours(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, legend=False, device=device)

### Figure 5: Uncontrolled Baseline, Overdamped, Inertial, ESH Proposals

In [ ]:
def load_uncontrolled_from_config(config_name):
    os.environ["RUN_CONFIG"] = config_name 
    with initialize(version_base=None, config_path="../config"):
        cfg = compose(config_name="nets_config")
    cfg.pinn_config.control = 'zero'
    cfg.pinn_config.record_every = 25
    return NETSModule(cfg=cfg.pinn_config).move_to_device(device)

# Create axes
fig, axes = plt.subplots(4, 5, figsize=(20, 15), dpi=500)
axes = axes.reshape((4, 5))
scale = 50.0
timesteps = torch.linspace(0,1,100).to(device)
num_samples = 200

def plot_pinn_proposal(pinn, axes):
    proposal = pinn.build_proposal(pinn.cfg.proposal).to(device)
    axes, plotted_timesteps = plot_proposal(
        proposal=proposal,
        num_samples=num_samples,
        ts=timesteps,
        scale=scale,
        axes=axes,
        use_alpha=False,
    )
    timeseries_plot_density(pinn.density_path.log_density, plotted_timesteps, scale=scale, axes=axes, alpha=0.5, vmin=-40, zorder=0, device=device)
    timeseries_plot_contours(pinn.density_path.log_density, plotted_timesteps, scale=scale, axes=axes, alpha=0.5, legend=False, levels=40, device=device)

# Load baseline
baseline_pinn = load_uncontrolled_from_config('learned_baseline')
plot_pinn_proposal(baseline_pinn, axes[0])

# Load overdamped
overdamped_pinn = load_uncontrolled_from_config('learned_overdamped')
plot_pinn_proposal(overdamped_pinn, axes[1])

# Load inertial
inertial_pinn = load_uncontrolled_from_config('learned_inertial')
plot_pinn_proposal(inertial_pinn, axes[2])

# Load ESH
esh_pinn = load_uncontrolled_from_config('learned_esh')
plot_pinn_proposal(esh_pinn, axes[3])

# Remove all ticks
for i in range(4):
    for j in range(5):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].set_title('')

# Set left-most ticks
for i in range(4):
    axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
    axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=8)

axes[0, 0].set_ylabel('Baseline', fontsize=15)
axes[1, 0].set_ylabel('Overdamped', fontsize=15)
axes[2, 0].set_ylabel('Inertial', fontsize=15)
axes[3, 0].set_ylabel('ESH', fontsize=15)

# Set bottom-most ticks
for i in range(5):
    tt = i / 4
    axes[3, i].set_xticks([-40, -20, 0, 20, 40])
    axes[3, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=8)
    axes[3, i].set_xlabel(f't={tt:.2f}', fontsize=15)

### Figure 6: Tempered Proposal Illustration

In [ ]:
# Load
os.environ["RUN_CONFIG"] = "learned_tempered" 
with initialize(version_base=None, config_path="../config"):
    cfg = compose(config_name="ctds_config")
    cfg.mt_pinn_config.mt_continuum = 'linear'
    cfg.mt_pinn_config.x_control = 'zero'
    cfg.mt_pinn_config.use_jarzynski = False # To speed up sampling

mt_pinn = CTDSModule(cfg=cfg.mt_pinn_config).move_to_device(device)

# Graph
mt_pinn.proposal.record_every = 125
t_bins = torch.linspace(0, 1, 5).to(device)
b_bins = torch.linspace(0.2, 1.0, 3).to(device)
scale = 50.0
sample_results, axes = mt_plot_proposal(
    proposal = mt_pinn.proposal,
    num_samples = 500,
    nts = 500,
    t_bins = t_bins,
    b_bins = b_bins,
    scale = scale
)

# Title the axes and set ticks to empty
for i in range(len(b_bins)):
    for j in range(len(t_bins)):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].set_title(axes[i, j].get_title(), fontsize=20)

# Set border ticks on temp axis
for i in range(len(b_bins)):
    axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
    axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=15)
    axes[i, 0].set_ylabel(r'$\beta=$' + f'{b_bins[i].item():.2f}', fontsize=25)

# Set border ticks on time axis
for i in range(len(t_bins)):
    axes[len(b_bins)-1, i].set_xticks([-40, -20, 0, 20, 40])
    axes[len(b_bins)-1, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=15)
    axes[len(b_bins)-1, i].set_xlabel(f't={t_bins[i].item():.2f}', fontsize=25)

mt_timeseries_plot_density(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, zorder=0, vmin=-40, device=device)
mt_timeseries_plot_contours(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, legend=False, device=device)

# Part 2: Model Results w/ Learned Path

In [3]:
# Learned path directory
learned_dir = '../pretrained_checkpoints/learned'

# Learned paths
learned_baseline_path = os.path.join(learned_dir, 'baseline.ckpt')

learned_overdamped_path = os.path.join(learned_dir, 'nets_overdamped.ckpt')
learned_overdamped_jarzynski_path = os.path.join(learned_dir, 'nets_overdamped_jarzynski.ckpt')

learned_inertial_path = os.path.join(learned_dir, 'nets_inertial.ckpt')
learned_inertial_jarzynski_path = os.path.join(learned_dir, 'nets_inertial_jarzynski.ckpt')

learned_esh_path = os.path.join(learned_dir, 'nets_esh.ckpt')

learned_ctds_path = os.path.join(learned_dir, 'ctds.ckpt')
learned_ctds_jarzynski_path = os.path.join(learned_dir, 'ctds_jarzynski.ckpt')
learned_ctds_esh_path = os.path.join(learned_dir, 'ctds_esh.ckpt')

### Figure 7: Learned Path Scatter

In [ ]:
from matplotlib import pyplot as plt

from src.models.nets import NETSModule
from src.models.ctds import CTDSModule
from src.simulation.dynamics import ForwardProcessSampleable
from src.utils.plotting import scatter_sampleable, plot_density, plot_contours
from src.systems.distributions import GMM


fig, axes = plt.subplots(3, 3, figsize=(20, 20), dpi=100)
axes = axes.reshape((3, 3))

model_ts = torch.linspace(0, 1, 100).to(device)
target = GMM.FAB_GMM(cov_scale=0.25).to(device)

# Plot density and contours on all subplots
for ax in axes.flatten():
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    plot_density(target.log_density, ax=ax, scale=50, alpha=0.5, vmin=-40, zorder=0, device=device)
    plot_contours(target.log_density, ax=ax, scale=50, alpha=0.5, legend=False, levels=50, zorder=0, device=device)

scatter_args = {
    'num_samples': 1000,
    'marker': 'o',
    's': 10,
    'color': 'black',
    'alpha': 0.5,
    'zorder': 1,
}

def scatter_from_path(path, title, ax, mt=False):
    if mt:
        pinn = CTDSModule.load_from_checkpoint(path).move_to_device(device)
    else:
        pinn = NETSModule.load_from_checkpoint(path).move_to_device(device)
    sampleable = ForwardProcessSampleable(
        forward_process=pinn.build_proposal("ode").dynamics,
        ts=model_ts
    ).to(device)
    ax.set_title(title, fontsize=18)
    scatter_sampleable(
        sampleable,
        ax=ax,
        **scatter_args,
    )

scatter_from_path(learned_baseline_path, 'Baseline', axes[0, 0])
scatter_from_path(learned_overdamped_path, 'NETS Overdamped', axes[0, 1])
scatter_from_path(learned_overdamped_jarzynski_path, 'NETS Overdamped w/ Jarzynski', axes[0, 2])
scatter_from_path(learned_inertial_path, 'NETS Inertial', axes[1, 0])
scatter_from_path(learned_inertial_jarzynski_path, 'NETS Inertial w/ Jarzynski', axes[1, 1])
scatter_from_path(learned_esh_path, 'NETS ESH', axes[1, 2])
scatter_from_path(learned_ctds_path, 'CTDS Inertial', axes[2, 0], mt=True)
scatter_from_path(learned_ctds_jarzynski_path, 'CTDS Inertial w/ Jarzynski', axes[2, 1], mt=True)
scatter_from_path(learned_ctds_esh_path, 'CTDS ESH', axes[2, 2], mt=True)

plt.show()

### Results 1: Learned Path

In [ ]:
import numpy as np
from tqdm import tqdm 

SAMPLES_PER_TRIAL = 2500
NUM_TRIALS = 10
NTS = 250

def compute_stats(pinn_module):
    """
    Compute the mean and variance of the proposal samples.
    """
    ode_proposal = pinn_module.build_proposal("ode")
    stats_dict = {}
    pbar = tqdm(range(NUM_TRIALS), desc="Computing statistics")
    for _ in pbar:
        ts = torch.linspace(0, 1, NTS).to(device).view(1, -1, 1).expand(SAMPLES_PER_TRIAL, -1, -1)
        metrics = ode_proposal.get_metrics(ts, label="ode")
        for key, value in metrics.items():
            if key not in stats_dict:
                stats_dict[key] = []
            stats_dict[key].append(value)
    
    stats_dict = {
        key: (np.mean(value), np.std(value)) for key, value in stats_dict.items()
    }

    # Print the statistics
    for key, (mean, std) in stats_dict.items():
        print(f"{key}: mean={mean:.2f}, std={std:.2f}")

def compute_stats_wrapper(path, title, mt = False):
    """
    Wrapper function to load the model and compute statistics.
    """
    if mt:
        pinn_module = CTDSModule.load_from_checkpoint(path).move_to_device(device)
    else: 
        pinn_module = NETSModule.load_from_checkpoint(path).move_to_device(device)
    print(f"Statistics for {title}:")
    compute_stats(pinn_module)

compute_stats_wrapper(learned_baseline_path, 'Baseline')
compute_stats_wrapper(learned_overdamped_path, 'Overdamped')
compute_stats_wrapper(learned_overdamped_jarzynski_path, 'Overdamped Jarzynski')
compute_stats_wrapper(learned_inertial_path, 'Inertial')
compute_stats_wrapper(learned_inertial_jarzynski_path, 'Inertial Jarzynski')
compute_stats_wrapper(learned_esh_path, 'ESH')
compute_stats_wrapper(learned_ctds_path, 'CTDS Inertial', mt=True)
compute_stats_wrapper(learned_ctds_jarzynski_path, 'CTDS Inertial Jarzynski', mt=True)
compute_stats_wrapper(learned_ctds_esh_path, 'CTDS ESH', mt=True)

### Figure 8: Tempered Proposal w/ Correctly Learned Free Energy

In [ ]:
mt_pinn = CTDSModule.load_from_checkpoint(learned_ctds_jarzynski_path).move_to_device(device)

# Graph
mt_pinn.proposal.record_every = 125
t_bins = torch.linspace(0, 1, 5).to(device)
b_bins = torch.linspace(0.2, 1.0, 3).to(device)
scale = 50.0
sample_results, axes = mt_plot_proposal(
    proposal = mt_pinn.proposal,
    num_samples = 500,
    nts = 500,
    t_bins = t_bins,
    b_bins = b_bins,
    scale = scale
)

# Title the axes and set ticks to empty
for i in range(len(b_bins)):
    for j in range(len(t_bins)):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].set_title(axes[i, j].get_title(), fontsize=20)

# Set border ticks on temp axis
for i in range(len(b_bins)):
    axes[i, 0].set_yticks([-40, -20, 0, 20, 40])
    axes[i, 0].set_yticklabels([-40, -20, 0, 20, 40], fontsize=15)
    axes[i, 0].set_ylabel(r'$\beta=$' + f'{b_bins[i].item():.2f}', fontsize=25)

# Set border ticks on time axis
for i in range(len(t_bins)):
    axes[len(b_bins)-1, i].set_xticks([-40, -20, 0, 20, 40])
    axes[len(b_bins)-1, i].set_xticklabels([-40, -20, 0, 20, 40], fontsize=15)
    axes[len(b_bins)-1, i].set_xlabel(f't={t_bins[i].item():.2f}', fontsize=25)

mt_timeseries_plot_density(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, zorder=0, vmin=-40, device=device)
mt_timeseries_plot_contours(mt_pinn.mt_continuum.log_density, ts=t_bins, bs=b_bins, scale=scale, axes=axes, alpha=0.5, legend=False, device=device)

### Comparison of Temperature Distribution of Proposal

In [ ]:
from hydra import compose, initialize
from matplotlib import pyplot as plt
import seaborn as sns

def get_bs(dynamics, converter, nts: int = 500, num_samples: int = 2500):
    """
    Sample from the CTDS dynamics.
    """
    ts = torch.linspace(0, 1, nts).to(device).view(1, -1, 1).expand(num_samples, -1, -1)
    xz_pxz, _ = dynamics.sample(
        ts=ts, num_samples=num_samples, use_tqdm = True
    )
    z = xz_pxz[:, 2]
    b = converter.xi_to_beta(z)
    return b

# Grab proposal from uninitialized model
os.environ["RUN_CONFIG"] = "learned_tempered" 
with initialize(version_base=None, config_path="../config"):
    cfg = compose(config_name="ctds_config")
    cfg.mt_pinn_config.x_control = 'zero'
    cfg.mt_pinn_config.use_jarzynski = False # To speed up sampling

untrained_ctds = CTDSModule(cfg=cfg.mt_pinn_config).move_to_device(device)
untrained_bs = get_bs(untrained_ctds.proposal.dynamics, untrained_ctds.converter).cpu().numpy()

# Grab proposal from initialized model
trained_ctds = CTDSModule.load_from_checkpoint(learned_ctds_jarzynski_path).move_to_device(device)
trained_bs = get_bs(trained_ctds.proposal.dynamics, trained_ctds.converter).cpu().numpy()   

In [ ]:
plt.figure(figsize=(8, 4), dpi=100)
sns.histplot(untrained_bs, color="skyblue", label="Untrained Inertial CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)
sns.histplot(trained_bs, color="orange", label="Inertial Trained Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

# Format x-axis and layout
plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of Inertial CTDS Proposal Samples at T=1.0")
plt.show()

# Part 3: Model Results w/ Linear Path

In [ ]:
# Learned path directory
linear_dir = '../pretrained_checkpoints/linear'

# Learned paths
linear_baseline_path = os.path.join(linear_dir, 'baseline.ckpt')

linear_overdamped_path = os.path.join(linear_dir, 'nets_overdamped.ckpt')
linear_overdamped_jarzynski_path = os.path.join(linear_dir, 'nets_overdamped_jarzynski.ckpt')

linear_inertial_path = os.path.join(linear_dir, 'nets_inertial.ckpt')
linear_inertial_jarzynski_path = os.path.join(linear_dir, 'nets_inertial_jarzynski.ckpt')

linear_esh_path = os.path.join(linear_dir, 'nets_esh.ckpt')

linear_ctds_path = os.path.join(linear_dir, 'ctds.ckpt')
linear_ctds_jarzynski_path = os.path.join(linear_dir, 'ctds_jarzynski.ckpt')
linear_ctds_esh_path = os.path.join(linear_dir, 'ctds_esh.ckpt')

### Figure 8: Linear Path Scatter

In [ ]:
from matplotlib import pyplot as plt

from src.models.nets import NETSModule
from src.models.ctds import CTDSModule
from src.simulation.dynamics import ForwardProcessSampleable
from src.utils.plotting import scatter_sampleable, plot_density, plot_contours
from src.systems.distributions import GMM


fig, axes = plt.subplots(3, 3, figsize=(20, 20), dpi=100)
axes = axes.reshape((3, 3))

model_ts = torch.linspace(0, 1, 100).to(device)
target = GMM.FAB_GMM(cov_scale=0.25).to(device)

# Plot density and contours on all subplots
for ax in axes.flatten():
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    plot_density(target.log_density, ax=ax, scale=50, alpha=0.5, vmin=-40, zorder=0, device=device)
    plot_contours(target.log_density, ax=ax, scale=50, alpha=0.5, legend=False, levels=50, zorder=0, device=device)

scatter_args = {
    'num_samples': 1000,
    'marker': 'o',
    's': 10,
    'color': 'black',
    'alpha': 0.5,
    'zorder': 1,
}

def scatter_from_path(path, title, ax, mt=False):
    if mt:
        pinn = CTDSModule.load_from_checkpoint(path).move_to_device(device)
    else:
        pinn = NETSModule.load_from_checkpoint(path).move_to_device(device)
    sampleable = ForwardProcessSampleable(
        forward_process=pinn.build_proposal("ode").dynamics,
        ts=model_ts
    ).to(device)
    ax.set_title(title, fontsize=18)
    scatter_sampleable(
        sampleable,
        ax=ax,
        **scatter_args,
    )

scatter_from_path(linear_baseline_path, 'Baseline', axes[0, 0])
scatter_from_path(linear_overdamped_path, 'NETS Overdamped', axes[0, 1])
scatter_from_path(linear_overdamped_jarzynski_path, 'NETS Overdamped w/ Jarzynski', axes[0, 2])
scatter_from_path(linear_inertial_path, 'NETS Inertial', axes[1, 0])
scatter_from_path(linear_inertial_jarzynski_path, 'NETS Inertial w/ Jarzynski', axes[1, 1])
scatter_from_path(linear_esh_path, 'NETS ESH', axes[1, 2])
scatter_from_path(linear_ctds_path, 'CTDS Inertial', axes[2, 0], mt=True)
scatter_from_path(linear_ctds_jarzynski_path, 'CTDS Inertial w/ Jarzynski', axes[2, 1], mt=True)
scatter_from_path(linear_ctds_esh_path, 'CTDS ESH', axes[2, 2], mt=True)

plt.show()

### Results 2: Linear Path

In [ ]:
import numpy as np
from tqdm import tqdm 

SAMPLES_PER_TRIAL = 2500
NUM_TRIALS = 10
NTS = 250

def compute_stats(pinn_module):
    """
    Compute the mean and variance of the proposal samples.
    """
    ode_proposal = pinn_module.build_proposal("ode")
    stats_dict = {}
    pbar = tqdm(range(NUM_TRIALS), desc="Computing statistics")
    for _ in pbar:
        ts = torch.linspace(0, 1, NTS).to(device).view(1, -1, 1).expand(SAMPLES_PER_TRIAL, -1, -1)
        metrics = ode_proposal.get_metrics(ts, label="ode")
        for key, value in metrics.items():
            if key not in stats_dict:
                stats_dict[key] = []
            stats_dict[key].append(value)
    
    stats_dict = {
        key: (np.mean(value), np.std(value)) for key, value in stats_dict.items()
    }

    # Print the statistics
    for key, (mean, std) in stats_dict.items():
        print(f"{key}: mean={mean:.2f}, std={std:.2f}")

def compute_stats_wrapper(path, title, mt = False):
    """
    Wrapper function to load the model and compute statistics.
    """
    if mt:
        pinn_module = CTDSModule.load_from_checkpoint(path).move_to_device(device)
    else: 
        pinn_module = NETSModule.load_from_checkpoint(path).move_to_device(device)
    print(f"Statistics for {title}:")
    compute_stats(pinn_module)

compute_stats_wrapper(linear_baseline_path, 'Baseline')
compute_stats_wrapper(linear_overdamped_path, 'Overdamped')
compute_stats_wrapper(linear_overdamped_jarzynski_path, 'Overdamped Jarzynski')
compute_stats_wrapper(linear_inertial_path, 'Inertial')
compute_stats_wrapper(linear_inertial_jarzynski_path, 'Inertial Jarzynski')
compute_stats_wrapper(linear_esh_path, 'ESH')
compute_stats_wrapper(linear_ctds_path, 'CTDS Inertial', mt=True)
compute_stats_wrapper(linear_ctds_jarzynski_path, 'CTDS Inertial Jarzynski', mt=True)
compute_stats_wrapper(linear_ctds_esh_path, 'CTDS ESH', mt=True)

### References
1. G. V. Steeg and A. Galstyan. Hamiltonian Dynamics with Non-Newtonian Momentum
for Rapid Sampling. 2021. arXiv: 2111.02434 [cs.LG]. url: https://arxiv.org/abs/2111.02434.